# 72 ? Experimento TF-IDF + DBSCAN

Este notebook testa agrupamento de descri??es normalizadas de itens de supermercado usando TF-IDF e DBSCAN. A an?lise usa dist?ncia cosseno, n?o usa NCM nem r?tulos humanos, e trata ru?do como resultado expl?cito do algoritmo. A amostra configur?vel mant?m o custo do c?lculo de vizinhan?a controlado. Execute de cima para baixo em uma sess?o limpa.


In [1]:
from hashlib import sha256
from datetime import datetime, timezone
from pathlib import Path
import logging
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn import config_context
from sklearn.cluster import DBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from matplotlib.colors import ListedColormap

from nofis_classifier.schema import DESCRIPTION_NORMALIZED, FREQUENCY

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook72")
pd.set_option("display.max_colwidth", 140)


## Configuracoes e comparabilidade

Controles comuns aos notebooks 70 a 76:

- Base completa `2025_NCMs_filtrados_semstopwords`, periodo 202501-202512.
- Ate 20.000 descricoes unicas para agrupamento, amostradas com semente 42 e ordenadas por descricao normalizada. Frequencia nao pondera o ajuste.
- Silhouette cosseno sobre uma amostra base fixa de ate 20.000 linhas, com semente 42. Ruido do algoritmo e excluido depois da amostragem; a quantidade efetivamente avaliada e registrada.
- Ranking exige silhouette definida e de 10 a 1.000 clusters efetivamente formados, sem contar ruido. Pesos: 0,75 para silhouette reescalada e 0,25 para cobertura do algoritmo.
- t-SNE 2D e 3D: mesmos ate 5.000 itens, semente 42, TruncatedSVD de ate 50 componentes, perplexidade 30, 1.000 iteracoes, init PCA (quando ha dimensoes suficientes), learning_rate auto e Barnes-Hut. As projecoes de representacoes diferentes sao exploratorias, nao um espaco comum de distancias.
- `comparison` em `run.json` registra os controles e o hash das descricoes/frequencias usadas no agrupamento.

TF-IDF: os mesmos 10 analyzers dos experimentos de texto, cruzados com `MIN_DF_VALUES = [1, 2, 5, 10]`, max_features 100.000, max_df 1.0, sublinear_tf True, norm l2 e dtype float64. Os vetores zerados sao registrados como diagnostico da representacao, separados do ruido do algoritmo.

DBSCAN: a mesma grade de eps e min_samples do notebook 73, preservando a uniao dos valores ja configurados nos dois. Grade atual: 40 representacoes x 11 valores de eps x 8 valores de min_samples = 3.520 ajustes.


In [2]:
DATASET_NAME = "2025_NCMs_filtrados_semstopwords"


PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Nao encontrei a raiz do repositorio com data/processed/.")
DATASET_DIR = PROJECT_ROOT / "data" / "processed" / DATASET_NAME
DATASET_FILES = sorted(DATASET_DIR.glob("*.parquet"))
if not DATASET_FILES:
    raise FileNotFoundError(f"Nenhum Parquet encontrado para {DATASET_NAME}: {DATASET_DIR}")
if len(DATASET_FILES) > 1:
    raise ValueError(
        f"Mais de um Parquet encontrado em {DATASET_DIR}; selecione o arquivo/periodo explicitamente: "
        f"{[path.name for path in DATASET_FILES]}"
    )
DATASET_PATH = DATASET_FILES[0]
if not DATASET_PATH.stem.startswith(f"{DATASET_NAME}-"):
    raise ValueError(
        f"O nome do arquivo deve comecar com '{DATASET_NAME}-' para derivar o periodo: "
        f"{DATASET_PATH.name}"
    )



PERIOD_LABEL = DATASET_PATH.stem[len(DATASET_NAME) + 1:]
MAX_DESCRIPTIONS = 20_000
CONSIDER_FREQUENCY_IN_CLUSTERING = False
SAMPLE_SEEDS = [42]
ANALYZER_CONFIGS = {'char_wb_2_4': ('char_wb', (2, 4)), 'char_wb_2_10': ('char_wb', (2, 10)), 'char_wb_3_5': ('char_wb', (3, 5)), 'char_wb_3_10': ('char_wb', (3, 10)), 'char_wb_4_8': ('char_wb', (4, 8)), 'char_wb_4_10': ('char_wb', (4, 10)), 'char_wb_5_10': ('char_wb', (5, 10)), 'word_1_3': ('word', (1, 3)), 'word_1_5': ('word', (1, 5)), 'word_2_5': ('word', (2, 5))}
MIN_DF_VALUES = [1, 2, 5, 10]
MAX_FEATURES = 100_000
SILHOUETTE_SAMPLE_SIZE = 20_000
DBSCAN_N_JOBS = 1
PAIRWISE_WORKING_MEMORY_MIB = 64
EPS_VALUES = [0.01, 0.02, 0.025, 0.03, 0.035, 0.04, 0.05, 0.07, 0.1, 0.5, 1]
MIN_SAMPLES_VALUES = [2, 3, 5, 7, 10, 20, 30, 100]
OUTPUT_ROOT = PROJECT_ROOT / Path("reports/tfidf_dbscan")

TSNE_MAX_POINTS = 5_000
TSNE_PERPLEXITY = 30
TSNE_MAX_ITER = 1_000
TSNE_RANDOM_STATE = 42
TSNE_3D_N_COMPONENTS = 3
TSNE_SVD_COMPONENTS = 50
METRIC_RANDOM_STATE = 42

MAX_DF = 1.0


### Configuracao do ranking final

`MIN_CLUSTER_FOR_RANKING = 10` define o minimo do ranking. O intervalo ate `MAX_CLUSTERS_FOR_RANKING` inclui os dois extremos e nao altera os ajustes da grade. Em DBSCAN/HDBSCAN, a contagem exclui o grupo de ruido (-1).


In [3]:
# O limite filtra apenas o ranking e a selecao final; nao altera os ajustes do DBSCAN.
MAX_CLUSTERS_FOR_RANKING = 1000
MIN_CLUSTER_FOR_RANKING = 10

# Os pesos definem a contribuicao relativa de silhouette e cobertura (1 - ruido).
# Com pesos iguais, as duas dimensoes contribuem igualmente para a metrica composta.
SILHOUETTE_WEIGHT = 0.75
NOISE_WEIGHT = 0.25


## 2. Fun??es da an?lise

A prepara??o remove descri??es vazias, mant?m uma linha por descri??o e valida que a frequ?ncia total n?o foi alterada. Cada configura??o ? executada em uma c?pia l?gica da mesma matriz TF-IDF.


In [4]:
def prepare_items(frame):
    initial_frequency = frame[FREQUENCY].sum()
    prepared = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].copy()
    prepared[DESCRIPTION_NORMALIZED] = prepared[DESCRIPTION_NORMALIZED].astype("string").str.strip()
    prepared = prepared[prepared[DESCRIPTION_NORMALIZED].notna() & prepared[DESCRIPTION_NORMALIZED].ne("")]
    prepared = prepared[prepared[FREQUENCY] >= 1].copy()
    prepared = (
        prepared.groupby(DESCRIPTION_NORMALIZED, as_index=False, sort=False)[FREQUENCY]
        .sum()
    )
    if prepared[FREQUENCY].sum() != initial_frequency:
        raise ValueError("A prepara??o alterou a soma de frequ?ncia.")
    return prepared.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)

def summarize_clusters(items, labels):
    result = items.copy()
    result["cluster"] = labels
    result["ruido"] = result["cluster"].eq(-1)
    summary = (
        result.groupby("cluster", as_index=False)
        .agg(descricoes_unicas=(DESCRIPTION_NORMALIZED, "size"), frequencia_total=(FREQUENCY, "sum"))
        .sort_values("descricoes_unicas", ascending=False)
    )
    return result, summary

def score_configuration(matrix, labels, sample_size, random_state):
    """Avalia a mesma amostra de linhas, excluindo ruido somente apos amostrar."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    cluster_count = len(np.unique(labels[~noise_mask]))
    base_count = min(sample_size, len(labels))
    rng = np.random.default_rng(random_state)
    positions = np.sort(rng.choice(len(labels), size=base_count, replace=False))
    positions = positions[~noise_mask[positions]]
    sample_labels = labels[positions]
    sample_cluster_count = len(np.unique(sample_labels))
    score = np.nan
    actual_count = len(positions)
    if 2 <= sample_cluster_count < actual_count:
        with config_context(working_memory=PAIRWISE_WORKING_MEMORY_MIB):
            score = float(silhouette_score(matrix[positions], sample_labels, metric="cosine"))
    else:
        actual_count = 0
    return {
        "cluster_count_excluding_noise": cluster_count,
        "noise_count": int(noise_mask.sum()),
        "noise_fraction": float(noise_mask.mean()),
        "silhouette_base_sample_size": base_count,
        "silhouette_sample_size": actual_count,
        "silhouette_cosine_without_noise": score,
    }


def compute_tsne_projection(features, max_points, perplexity, max_iter, random_state, n_components=2):
    """Usa a mesma amostra, SVD e parametros t-SNE em todas as representacoes."""
    point_count = features.shape[0]
    sample_count = min(point_count, max_points)
    if sample_count < 3 or n_components not in (2, 3):
        raise ValueError("t-SNE exige pelo menos tres itens e n_components 2 ou 3.")
    rng = np.random.default_rng(random_state)
    indices = np.sort(rng.choice(point_count, size=sample_count, replace=False))
    sampled = features[indices]
    dimensions = min(TSNE_SVD_COMPONENTS, sampled.shape[1] - 1, sample_count - 1)
    if dimensions >= 1:
        reduced = TruncatedSVD(n_components=dimensions, random_state=random_state).fit_transform(sampled)
    else:
        reduced = sampled.toarray() if hasattr(sampled, "toarray") else np.asarray(sampled)
        dimensions = reduced.shape[1]
    coordinates = TSNE(
        n_components=n_components,
        perplexity=min(float(perplexity), (sample_count - 1) / 3),
        init="pca" if dimensions >= n_components else "random",
        learning_rate="auto", max_iter=max_iter, random_state=random_state,
        method="barnes_hut",
    ).fit_transform(reduced)
    return indices, coordinates, dimensions


def save_tsne_plot(coordinates, labels, run_id, output_path):
    """Salva um PNG t-SNE colorido pelos clusters, com o ru?do em cinza."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    cluster_ids = np.unique(labels[~noise_mask])
    if cluster_ids.size:
        palette = plt.get_cmap("tab20").colors
        cluster_cmap = ListedColormap([
            palette[index % len(palette)] for index in range(len(cluster_ids))
        ])
        cluster_codes = np.searchsorted(cluster_ids, labels[~noise_mask])
    else:
        cluster_cmap = None
        cluster_codes = np.empty(0, dtype=int)

    figure, axis = plt.subplots(figsize=(11, 8), constrained_layout=True)
    if noise_mask.any():
        axis.scatter(
            coordinates[noise_mask, 0], coordinates[noise_mask, 1],
            c="#b8b8b8", s=8, alpha=0.3, linewidths=0,
        )
    if cluster_ids.size:
        axis.scatter(
            coordinates[~noise_mask, 0], coordinates[~noise_mask, 1],
            c=cluster_codes, cmap=cluster_cmap, vmin=-0.5,
            vmax=len(cluster_ids) - 0.5, s=9, alpha=0.75, linewidths=0,
        )
    axis.set_title(
        f"t-SNE: {run_id}\n{len(cluster_ids)} clusters; "
        f"{int(noise_mask.sum()):,} descri??es como ru?do"
    )
    axis.set_xlabel("t-SNE 1")
    axis.set_ylabel("t-SNE 2")
    axis.text(
        0.01, 0.01,
        "Ru?do em cinza; as cores se repetem a cada 20 clusters.",
        transform=axis.transAxes, fontsize=8, color="#555555",
    )
    figure.savefig(output_path, dpi=160)
    plt.close(figure)


def save_tsne_3d_html(coordinates, labels, run_id, output_path):
    """Salva uma visualiza??o t-SNE 3D interativa e autocontida em HTML."""
    points = [
        {"x": float(x), "y": float(y), "z": float(z), "label": int(label)}
        for (x, y, z), label in zip(coordinates, labels)
    ]
    data_json = json.dumps(points, separators=(",", ":"))
    title = str(run_id).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    page = r"""<!doctype html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>t-SNE 3D: __TITLE__</title>
<style>
html, body { margin: 0; height: 100%; background: #f7f8fa; color: #202124; font: 14px system-ui, sans-serif; }
main { display: flex; flex-direction: column; height: 100%; }
header { padding: 12px 18px; background: white; border-bottom: 1px solid #ddd; }
h1 { margin: 0 0 4px; font-size: 17px; }
p { margin: 0; color: #555; }
canvas { display: block; width: 100%; flex: 1; min-height: 300px; cursor: grab; }
canvas:active { cursor: grabbing; }
</style>
</head>
<body>
<main>
<header><h1>Proje??o t-SNE 3D: __TITLE__</h1><p>Arraste para girar; use a roda do mouse para aproximar. Ru?do em cinza.</p></header>
<canvas id="plot"></canvas>
</main>
<script>
const points = __POINTS__;
const canvas = document.getElementById("plot");
const ctx = canvas.getContext("2d");
let yaw = 0.65, pitch = -0.35, zoom = 1, dragging = false, lastX = 0, lastY = 0;
const colors = ["#3366cc", "#dc3912", "#ff9900", "#109618", "#990099", "#0099c6", "#dd4477", "#66aa00", "#b82e2e", "#316395", "#994499", "#22aa99", "#aaaa11", "#6633cc", "#e67300", "#8b0707", "#651067", "#329262", "#5574a6", "#3b3eac"];
const bounds = ["x", "y", "z"].map(key => {
  const values = points.map(point => point[key]);
  const low = Math.min(...values), high = Math.max(...values);
  return { low, span: Math.max(high - low, 1e-9) };
});
function resize() {
  const ratio = window.devicePixelRatio || 1;
  canvas.width = Math.max(1, Math.floor(canvas.clientWidth * ratio));
  canvas.height = Math.max(1, Math.floor(canvas.clientHeight * ratio));
  ctx.setTransform(ratio, 0, 0, ratio, 0, 0);
  draw();
}
function draw() {
  const width = canvas.clientWidth, height = canvas.clientHeight;
  ctx.clearRect(0, 0, width, height);
  const scale = Math.min(width, height) * 0.7 * zoom;
  const cy = Math.cos(yaw), sy = Math.sin(yaw), cp = Math.cos(pitch), sp = Math.sin(pitch);
  const projected = points.map(point => {
    const x = ((point.x - bounds[0].low) / bounds[0].span - 0.5) * 2;
    const y = ((point.y - bounds[1].low) / bounds[1].span - 0.5) * 2;
    const z = ((point.z - bounds[2].low) / bounds[2].span - 0.5) * 2;
    const rx = x * cy - z * sy, rz = x * sy + z * cy;
    const ry = y * cp - rz * sp, depth = y * sp + rz * cp;
    return { x: width / 2 + rx * scale / 2, y: height / 2 - ry * scale / 2, depth, label: point.label };
  }).sort((a, b) => a.depth - b.depth);
  for (const point of projected) {
    ctx.globalAlpha = point.label === -1 ? 0.24 : 0.78;
    ctx.fillStyle = point.label === -1 ? "#999999" : colors[Math.abs(point.label) % colors.length];
    ctx.beginPath();
    ctx.arc(point.x, point.y, point.label === -1 ? 2 : 3, 0, Math.PI * 2);
    ctx.fill();
  }
  ctx.globalAlpha = 1;
}
canvas.addEventListener("pointerdown", event => { dragging = true; lastX = event.clientX; lastY = event.clientY; canvas.setPointerCapture(event.pointerId); });
canvas.addEventListener("pointerup", () => { dragging = false; });
canvas.addEventListener("pointermove", event => {
  if (!dragging) return;
  yaw += (event.clientX - lastX) * 0.008;
  pitch = Math.max(-1.5, Math.min(1.5, pitch + (event.clientY - lastY) * 0.008));
  lastX = event.clientX; lastY = event.clientY; draw();
});
canvas.addEventListener("wheel", event => { event.preventDefault(); zoom = Math.max(0.3, Math.min(5, zoom * (event.deltaY < 0 ? 1.1 : 0.9))); draw(); }, { passive: false });
window.addEventListener("resize", resize);
resize();
</script>
</body>
</html>"""
    page = page.replace("__TITLE__", title).replace("__POINTS__", data_json)
    output_path.write_text(page, encoding="utf-8")


def comparison_metadata(frame):
    """Registra controles comuns e a identidade exata da amostra de agrupamento."""
    sample = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
    return {
        "dataset_name": DATASET_NAME,
        "training_description_count": len(sample),
        "training_sample_sha256": sha256(pd.util.hash_pandas_object(sample, index=False).values.tobytes()).hexdigest(),
        "max_descriptions": MAX_DESCRIPTIONS, "sample_seed": 42,
        "frequency_weighted": False,
        "silhouette_metric": "cosine", "silhouette_sample_size": SILHOUETTE_SAMPLE_SIZE,
        "metric_random_state": METRIC_RANDOM_STATE,
        "metric_sampling": "fixed rows, then exclude algorithmic noise",
        "min_cluster_for_ranking": MIN_CLUSTER_FOR_RANKING,
        "max_clusters_for_ranking": MAX_CLUSTERS_FOR_RANKING,
        "silhouette_weight": SILHOUETTE_WEIGHT, "noise_weight": NOISE_WEIGHT,
        "tsne": {
            "max_points": TSNE_MAX_POINTS, "perplexity": TSNE_PERPLEXITY,
            "max_iter": TSNE_MAX_ITER, "random_state": TSNE_RANDOM_STATE,
            "reduction": "TruncatedSVD", "svd_components": TSNE_SVD_COMPONENTS,
            "dimensions": [2, 3], "init": "pca", "learning_rate": "auto",
            "method": "barnes_hut",
        },
    }


## 3. Carregar e preparar dados

Origem: Parquet produzido pelos notebooks de preparacao de dados do projeto, com descricoes de supermercado, NCMs e duplicatas agregadas por descricao. A amostragem e feita uma vez por semente e reutilizada entre os analyzers daquela semente.


In [5]:
raw_items = pd.read_parquet(DATASET_PATH)
required_columns = {DESCRIPTION_NORMALIZED, FREQUENCY}
missing_columns = required_columns - set(raw_items.columns)
if missing_columns:
    raise ValueError(f"Colunas ausentes no dataset: {sorted(missing_columns)}")
base_items = prepare_items(raw_items)
logger.info("Descricoes disponiveis apos preparacao: %d", len(base_items))


2026-10-24 18:25:42,879 INFO Descricoes disponiveis apos preparacao: 146999


## 4. Vetorizar, ajustar DBSCAN e comparar parametros

Cada combinacao de semente e analyzer gera uma matriz TF-IDF propria. `metric="cosine"` permite trabalhar diretamente com a matriz esparsa. O ruido (`cluster = -1`) e excluido da silhouette e contabilizado separadamente; configuracoes com menos de dois clusters validos ficam sem silhouette. A matriz completa nao e convertida para densa.

Para visualizacao, uma amostra fixa de ate `TSNE_MAX_POINTS` descricoes e reduzida com TruncatedSVD e projetada uma vez por representacao. As coordenadas sao reutilizadas para salvar um PNG por combinacao de parametros.

A proje??o t-SNE 3D ? calculada separadamente e cada configura??o tamb?m ? salva como HTML interativo autocontido em `tsne_3d_html/`.


In [6]:
logger.info("Grade planejada: %d ajustes", len(SAMPLE_SEEDS) * len(ANALYZER_CONFIGS) * len(MIN_DF_VALUES) * len(EPS_VALUES) * len(MIN_SAMPLES_VALUES))

records = []
labels_by_run = {}
items_by_run = {}
matrices_by_representation = {}
items_by_seed = {}

for sample_seed in SAMPLE_SEEDS:
    sampled_items = base_items
    if MAX_DESCRIPTIONS is not None and len(base_items) > MAX_DESCRIPTIONS:
        sampled_items = base_items.sample(n=MAX_DESCRIPTIONS, random_state=sample_seed)
    sampled_items = sampled_items.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
    items_by_seed[sample_seed] = sampled_items

    for representation, (analyzer, ngram_range) in ANALYZER_CONFIGS.items():
        for min_df in MIN_DF_VALUES:
            logger.info(
                "Vetorizando analyzer=%s, min_df=%d, seed=%d, descricoes=%d",
                analyzer, min_df, sample_seed, len(sampled_items),
            )
            vectorizer = TfidfVectorizer(
                analyzer=analyzer, ngram_range=ngram_range, min_df=min_df,
                max_features=MAX_FEATURES, max_df=MAX_DF, sublinear_tf=True, norm="l2", dtype=np.float64,
            )
            matrix = vectorizer.fit_transform(sampled_items[DESCRIPTION_NORMALIZED].astype(str))
            zero_vector_count = int((matrix.getnnz(axis=1) == 0).sum())
            # Mantem as matrizes esparsas para gerar as projecoes em uma celula separada.
            matrices_by_representation[(sample_seed, representation, min_df)] = matrix
            logger.info(
                "Analyzer=%s min_df=%d seed=%d: %d termos, %d vetores zerados",
                analyzer, min_df, sample_seed, matrix.shape[1], zero_vector_count,
            )

            for eps in EPS_VALUES:
                for min_samples in MIN_SAMPLES_VALUES:
                    run_id = f"seed{sample_seed}_{representation}_min_df{min_df}_eps{eps:g}_min_samples{min_samples}"
                    logger.info("Ajustando DBSCAN %s", run_id)
                    model = DBSCAN(
                        eps=eps, min_samples=min_samples, metric="cosine",
                        algorithm="brute", n_jobs=DBSCAN_N_JOBS,
                    )
                    sample_weight = (
                        sampled_items[FREQUENCY].to_numpy(dtype=float)
                        if CONSIDER_FREQUENCY_IN_CLUSTERING else None
                    )
                    with config_context(working_memory=PAIRWISE_WORKING_MEMORY_MIB):
                        model.fit(matrix, sample_weight=sample_weight)
                    labels = model.labels_
                    with config_context(working_memory=PAIRWISE_WORKING_MEMORY_MIB):
                        metrics = score_configuration(
                            matrix, labels, SILHOUETTE_SAMPLE_SIZE,
                            METRIC_RANDOM_STATE,
                        )
                    records.append({
                        "run_id": run_id, "sample_seed": sample_seed,
                        "representation": representation, "analyzer": analyzer,
                        "ngram_range": str(ngram_range), "min_df": min_df,
                        "feature_count": matrix.shape[1], "zero_vector_count": zero_vector_count,
                        "eps": eps, "min_samples": min_samples,
                        "frequency_weighted": CONSIDER_FREQUENCY_IN_CLUSTERING, **metrics,
                    })
                    labels_by_run[run_id] = labels
                    items_by_run[run_id] = sampled_items

# Mantem todas as metricas para persistencia; o ranking e calculado em uma celula separada.
results = pd.DataFrame(records)


2026-10-24 18:25:42,892 INFO Grade planejada: 3520 ajustes


2026-10-24 18:25:42,915 INFO Vetorizando analyzer=char_wb, min_df=1, seed=42, descricoes=20000


2026-10-24 18:25:43,567 INFO Analyzer=char_wb min_df=1 seed=42: 36881 termos, 0 vetores zerados


2026-10-24 18:25:43,567 INFO Ajustando DBSCAN seed42_char_wb_2_4_min_df1_eps0.01_min_samples2


2026-10-24 18:25:50,213 INFO Ajustando DBSCAN seed42_char_wb_2_4_min_df1_eps0.01_min_samples3


...

2026-10-25 00:14:19,225 INFO Ajustando DBSCAN seed42_word_2_5_min_df10_eps1_min_samples7


2026-10-25 00:14:29,652 INFO Ajustando DBSCAN seed42_word_2_5_min_df10_eps1_min_samples10


2026-10-25 00:14:40,138 INFO Ajustando DBSCAN seed42_word_2_5_min_df10_eps1_min_samples20


2026-10-25 00:14:50,707 INFO Ajustando DBSCAN seed42_word_2_5_min_df10_eps1_min_samples30


2026-10-25 00:15:01,118 INFO Ajustando DBSCAN seed42_word_2_5_min_df10_eps1_min_samples100


### Calcular a metrica composta e ordenar os resultados

A silhouette e reescalada para 0 a 1 e combinada com a proporcao de descricoes classificadas (`1 - noise_fraction`). O score composto determina a ordem; silhouette e ruido servem apenas de desempate quando os scores forem iguais.


In [7]:
if not 1 <= MIN_CLUSTER_FOR_RANKING <= MAX_CLUSTERS_FOR_RANKING:
    raise ValueError(
        "Os limites devem atender a 1 <= MIN_CLUSTER_FOR_RANKING <= MAX_CLUSTERS_FOR_RANKING."
    )

if SILHOUETTE_WEIGHT < 0 or NOISE_WEIGHT < 0 or SILHOUETTE_WEIGHT + NOISE_WEIGHT <= 0:
    raise ValueError("Os pesos devem ser nao negativos e pelo menos um deles deve ser maior que zero.")

# Silhouette varia de -1 a 1; reescale para 0 a 1 para combinar com cobertura.
results["silhouette_quality"] = (results["silhouette_cosine_without_noise"] + 1) / 2
results["noise_quality"] = 1 - results["noise_fraction"]
weight_total = SILHOUETTE_WEIGHT + NOISE_WEIGHT
results["composite_score"] = (
    SILHOUETTE_WEIGHT * results["silhouette_quality"]
    + NOISE_WEIGHT * results["noise_quality"]
) / weight_total

# O score composto e o criterio principal; os outros criterios so desempataram scores iguais.
ranking = (
    results.loc[
        results["cluster_count_excluding_noise"].between(
            MIN_CLUSTER_FOR_RANKING, MAX_CLUSTERS_FOR_RANKING, inclusive="both",
        ) & results["silhouette_cosine_without_noise"].notna()
    ]
    .sort_values(
        ["composite_score", "silhouette_cosine_without_noise", "noise_fraction"],
        ascending=[False, False, True], na_position="last",
    )
    .reset_index(drop=True)
)
if ranking.empty:
    raise ValueError(
        f"Nenhum resultado atende ao intervalo de {MIN_CLUSTER_FOR_RANKING} a "
        f"{MAX_CLUSTERS_FOR_RANKING} clusters. Ajuste eps/min_samples ou o limite."
    )
# Explicita a posicao de cada configuracao depois dos filtros e criterios acima.
ranking.insert(0, "rank", range(1, len(ranking) + 1))


In [8]:
display(ranking)

,rank,run_id,sample_seed,representation,analyzer,ngram_range,min_df,feature_count,zero_vector_count,eps,...,frequency_weighted,cluster_count_excluding_noise,noise_count,noise_fraction,silhouette_base_sample_size,silhouette_sample_size,silhouette_cosine_without_noise,silhouette_quality,noise_quality,composite_score
0,1,seed42_word_2_5_min_df10_eps0.025_min_samples3,42,word_2_5,word,"(2, 5)",10,930,10796,0.025,...,False,734,12783,0.63915,20000,7217,0.999630,0.999815,0.36085,0.840074
1,2,seed42_word_2_5_min_df10_eps0.01_min_samples3,42,word_2_5,word,"(2, 5)",10,930,10796,0.010,...,False,734,12792,0.63960,20000,7208,0.999859,0.999929,0.36040,0.840047
2,3,seed42_word_2_5_min_df10_eps0.03_min_samples3,42,word_2_5,word,"(2, 5)",10,930,10796,0.030,...,False,734,12781,0.63905,20000,7219,0.999473,0.999737,0.36095,0.840040
3,4,seed42_word_2_5_min_df10_eps0.02_min_samples3,42,word_2_5,word,"(2, 5)",10,930,10796,0.020,...,False,734,12788,0.63940,20000,7212,0.999681,0.999840,0.36060,0.840030
4,5,seed42_word_2_5_min_df10_eps0.05_min_samples3,42,word_2_5,word,"(2, 5)",10,930,10796,0.050,...,False,734,12753,0.63765,20000,7247,0.998388,0.999194,0.36235,0.839983
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1280,1281,seed42_char_wb_4_10_min_df1_eps0.5_min_samples30,42,char_wb_4_10,char_wb,"(4, 10)",1,100000,0,0.500,...,False,27,11876,0.59380,20000,8124,0.031995,0.515998,0.40620,0.488548
1281,1282,seed42_char_wb_5_10_min_df1_eps0.5_min_samples30,42,char_wb_5_10,char_wb,"(5, 10)",1,100000,1,0.500,...,False,20,11623,0.58115,20000,8377,0.019892,0.509946,0.41885,0.487172
1282,1283,seed42_word_2_5_min_df10_eps0.5_min_samples20,42,word_2_5,word,"(2, 5)",10,930,10796,0.500,...,False,29,11972,0.59860,20000,8028,0.031036,0.515518,0.40140,0.486988
1283,1284,seed42_word_2_5_min_df10_eps0.5_min_samples30,42,word_2_5,word,"(2, 5)",10,930,10796,0.500,...,False,26,13883,0.69415,20000,6117,0.054613,0.527307,0.30585,0.471942


In [9]:
ranking.head(20)

,rank,run_id,sample_seed,representation,analyzer,ngram_range,min_df,feature_count,zero_vector_count,eps,...,frequency_weighted,cluster_count_excluding_noise,noise_count,noise_fraction,silhouette_base_sample_size,silhouette_sample_size,silhouette_cosine_without_noise,silhouette_quality,noise_quality,composite_score
0,1,seed42_word_2_5_min_df10_eps0.025_min_samples3,42,word_2_5,word,"(2, 5)",10,930,10796,0.025,...,False,734,12783,0.63915,20000,7217,0.999630,0.999815,0.36085,0.840074
1,2,seed42_word_2_5_min_df10_eps0.01_min_samples3,42,word_2_5,word,"(2, 5)",10,930,10796,0.010,...,False,734,12792,0.63960,20000,7208,0.999859,0.999929,0.36040,0.840047
2,3,seed42_word_2_5_min_df10_eps0.03_min_samples3,42,word_2_5,word,"(2, 5)",10,930,10796,0.030,...,False,734,12781,0.63905,20000,7219,0.999473,0.999737,0.36095,0.840040
3,4,seed42_word_2_5_min_df10_eps0.02_min_samples3,42,word_2_5,word,"(2, 5)",10,930,10796,0.020,...,False,734,12788,0.63940,20000,7212,0.999681,0.999840,0.36060,0.840030
4,5,seed42_word_2_5_min_df10_eps0.05_min_samples3,42,word_2_5,word,"(2, 5)",10,930,10796,0.050,...,False,734,12753,0.63765,20000,7247,0.998388,0.999194,0.36235,0.839983
5,6,seed42_word_2_5_min_df10_eps0.07_min_samples3,42,word_2_5,word,"(2, 5)",10,930,10796,0.070,...,False,737,12729,0.63645,20000,7271,0.997231,0.998616,0.36355,0.839849
6,7,seed42_word_2_5_min_df10_eps0.035_min_samples3,42,word_2_5,word,"(2, 5)",10,930,10796,0.035,...,False,736,12768,0.63840,20000,7232,0.998382,0.999191,0.36160,0.839793
7,8,seed42_word_2_5_min_df10_eps0.04_min_samples3,42,word_2_5,word,"(2, 5)",10,930,10796,0.040,...,False,735,12763,0.63815,20000,7237,0.998174,0.999087,0.36185,0.839778
8,9,seed42_word_2_5_min_df10_eps0.1_min_samples3,42,word_2_5,word,"(2, 5)",10,930,10796,0.100,...,False,745,12654,0.63270,20000,7346,0.992096,0.996048,0.36730,0.838861
9,10,seed42_word_2_5_min_df10_eps0.03_min_samples5,42,word_2_5,word,"(2, 5)",10,930,10796,0.030,...,False,514,13517,0.67585,20000,6483,0.999926,0.999963,0.32415,0.831010


## 5. Revisar um resultado e salvar relat?rios

Escolha `SELECTED_RUN_ID` ap?s comparar quantidade de clusters, ru?do e silhouette. A tabela de resumo inclui `-1` como grupo de ru?do. Os arquivos s?o gravados em uma pasta com identificador UTC ?nico para preservar execu??es anteriores.

Os graficos ficam em `tsne_plots/`, um PNG por combinacao, e as coordenadas da amostra em `tsne_coordinates/`, dentro da pasta da execucao.


In [10]:
SELECTED_RUN_ID = ranking.iloc[0]["run_id"]
selected = results.set_index("run_id").loc[SELECTED_RUN_ID]
selected_items = items_by_run[SELECTED_RUN_ID]
selected_labels = labels_by_run[SELECTED_RUN_ID]
assignments, cluster_summary = summarize_clusters(selected_items, selected_labels)
cluster_summary = cluster_summary.sort_values("frequencia_total", ascending=False).reset_index(drop=True)
cluster_sizes = assignments.loc[assignments["cluster"].ne(-1)].groupby("cluster")[DESCRIPTION_NORMALIZED].size()
singleton_clusters = cluster_sizes[cluster_sizes.eq(1)].index
frequent_singletons = assignments[assignments["cluster"].isin(singleton_clusters)].sort_values(
    FREQUENCY, ascending=False
)
frequent_noise = assignments[assignments["cluster"].eq(-1)].sort_values(
    FREQUENCY, ascending=False
)
display(cluster_summary.head(20))
display(frequent_singletons.head(20))
display(frequent_noise.head(20))

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ%f")
output_dir = OUTPUT_ROOT / f"{PERIOD_LABEL}_{RUN_ID}"
output_dir.mkdir(parents=True, exist_ok=False)
results.to_csv(output_dir / "metrics.csv", index=False)
ranking.to_csv(output_dir / "ranking.csv", index=False)
assignments.to_csv(output_dir / "assignments.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv(output_dir / "cluster_summary.csv", index=False, encoding="utf-8-sig")
frequent_singletons.to_csv(
    output_dir / "frequent_singleton_descriptions.csv", index=False, encoding="utf-8-sig"
)
frequent_noise.to_csv(
    output_dir / "frequent_noise_descriptions.csv", index=False, encoding="utf-8-sig"
)

,cluster,descricoes_unicas,frequencia_total
0,-1,12783,63659
1,18,20,3622
2,246,13,1952
3,101,71,1669
4,264,29,1543
5,609,34,1164
6,437,59,1061
7,402,11,953
8,211,31,942
9,332,15,901


,descricao_normalizada,frequencia,cluster,ruido


,descricao_normalizada,frequencia,cluster,ruido
10531,jantar,1493,-1,True
18043,salsicha,1339,-1,True
18070,salsicha kg,906,-1,True
13101,multiuso 500ml ye advir,871,-1,True
1727,apagador quadro branco go office,805,-1,True
4572,caldo galinha 1 kg,717,-1,True
16199,refeicao,612,-1,True
15582,prancheta eucatex oficio c 1 un,574,-1,True
4904,ceia,567,-1,True
17795,sal,540,-1,True


## 6. Gerar visualizacoes t-SNE 2D e 3D

Esta etapa roda depois do ajuste e do ranking. Ela projeta as matrizes TF-IDF ja calculadas e salva as coordenadas e os graficos, sem executar novamente o DBSCAN.


In [11]:
# Calcula as proje??es 2D existentes e uma proje??o 3D separada por representa??o.
tsne_by_representation = {}
tsne_3d_by_representation = {}
for (sample_seed, representation, min_df), matrix in matrices_by_representation.items():
    logger.info("Calculando t-SNE 2D para %s, min_df=%d, seed=%d", representation, min_df, sample_seed)
    tsne_indices, tsne_coordinates, tsne_dimensions = compute_tsne_projection(
        matrix, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
        TSNE_RANDOM_STATE,
    )
    tsne_by_representation[(sample_seed, representation, min_df)] = {
        "indices": tsne_indices, "coordinates": tsne_coordinates,
        "reduced_dimensions": tsne_dimensions,
    }

    logger.info("Calculando t-SNE 3D para %s, min_df=%d, seed=%d", representation, min_df, sample_seed)
    tsne_3d_indices, tsne_3d_coordinates, tsne_3d_dimensions = compute_tsne_projection(
        matrix, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
        TSNE_RANDOM_STATE, n_components=TSNE_3D_N_COMPONENTS,
    )
    if not np.array_equal(tsne_indices, tsne_3d_indices):
        raise ValueError("As proje??es 2D e 3D devem usar a mesma amostra de descri??es.")
    tsne_3d_by_representation[(sample_seed, representation, min_df)] = {
        "indices": tsne_3d_indices, "coordinates": tsne_3d_coordinates,
        "reduced_dimensions": tsne_3d_dimensions,
    }

# Salva coordenadas 2D e um PNG por configura??o.
tsne_coordinates_dir = output_dir / "tsne_coordinates"
tsne_plots_dir = output_dir / "tsne_plots"
tsne_3d_html_dir = output_dir / "tsne_3d_html"
tsne_coordinates_dir.mkdir()
tsne_plots_dir.mkdir()
tsne_3d_html_dir.mkdir()
tsne_artifacts = {}
for (sample_seed, representation, min_df), projection in tsne_by_representation.items():
    indices = projection["indices"]
    coordinates = projection["coordinates"]
    coordinate_frame = items_by_seed[sample_seed].iloc[indices][
        [DESCRIPTION_NORMALIZED, FREQUENCY]
    ].reset_index(drop=True)
    coordinate_frame["tsne_1"] = coordinates[:, 0]
    coordinate_frame["tsne_2"] = coordinates[:, 1]
    coordinate_name = f"seed{sample_seed}_{representation}_min_df{min_df}_coordinates.csv"
    coordinate_frame.to_csv(
        tsne_coordinates_dir / coordinate_name, index=False, encoding="utf-8-sig",
    )
    tsne_artifacts[f"seed{sample_seed}_{representation}_min_df{min_df}"] = {
        "coordinates_file": f"tsne_coordinates/{coordinate_name}",
        "sample_size": len(indices),
        "reduced_dimensions": projection["reduced_dimensions"],
    }

for run_id, labels in labels_by_run.items():
    result_row = results.loc[results["run_id"].eq(run_id)].iloc[0]
    representation_key = (
        int(result_row["sample_seed"]), result_row["representation"], int(result_row["min_df"]),
    )
    projection = tsne_by_representation[representation_key]
    projection_3d = tsne_3d_by_representation[representation_key]
    plot_labels = np.asarray(labels)[projection["indices"]]
    plot_path = tsne_plots_dir / f"{run_id}.png"
    save_tsne_plot(projection["coordinates"], plot_labels, run_id, plot_path)
    tsne_artifacts[f"plot:{run_id}"] = str(plot_path.relative_to(output_dir))

    plot_3d_path = tsne_3d_html_dir / f"{run_id}.html"
    save_tsne_3d_html(projection_3d["coordinates"], plot_labels, run_id, plot_3d_path)
    tsne_artifacts[f"plot_3d:{run_id}"] = str(plot_3d_path.relative_to(output_dir))

logger.info("Imagens t-SNE 2D e gr?ficos HTML 3D salvos: %d", len(labels_by_run))

run_metadata = {
    "comparison": comparison_metadata(items_by_seed[SAMPLE_SEEDS[0]]),
    "dataset_name": DATASET_NAME, "period": PERIOD_LABEL,
    "frequency_weighted": CONSIDER_FREQUENCY_IN_CLUSTERING,
    "dataset_path": str(DATASET_PATH), "max_descriptions": MAX_DESCRIPTIONS,
    "sample_seeds": SAMPLE_SEEDS, "analyzer_configs": {
        representation: {"analyzer": analyzer, "ngram_range": list(ngram_range)}
        for representation, (analyzer, ngram_range) in ANALYZER_CONFIGS.items()
    },
    "min_df_values": MIN_DF_VALUES, "max_features": MAX_FEATURES,
    "selected_run_id": SELECTED_RUN_ID,
    "eps_values": EPS_VALUES, "min_samples_values": MIN_SAMPLES_VALUES,
    "silhouette_sample_size": SILHOUETTE_SAMPLE_SIZE,
    "pairwise_working_memory_mib": PAIRWISE_WORKING_MEMORY_MIB,
    "max_clusters_for_ranking": MAX_CLUSTERS_FOR_RANKING,
    "min_cluster_for_ranking": MIN_CLUSTER_FOR_RANKING,
    "silhouette_weight": SILHOUETTE_WEIGHT, "noise_weight": NOISE_WEIGHT,
    "tsne": {
        "max_points": TSNE_MAX_POINTS, "perplexity": TSNE_PERPLEXITY,
        "max_iter": TSNE_MAX_ITER, "random_state": TSNE_RANDOM_STATE,
        "svd_components": TSNE_SVD_COMPONENTS,
        "three_dimensional_components": TSNE_3D_N_COMPONENTS,
        "artifacts": tsne_artifacts,
    },
}
(output_dir / "run.json").write_text(
    json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
logger.info("Relatorios e imagens t-SNE salvos em %s", output_dir)


2026-10-25 00:15:12,357 INFO Calculando t-SNE 2D para char_wb_2_4, min_df=1, seed=42


2026-10-25 00:15:21,116 INFO Calculando t-SNE 3D para char_wb_2_4, min_df=1, seed=42


2026-10-25 00:15:41,825 INFO Calculando t-SNE 2D para char_wb_2_4, min_df=2, seed=42


2026-10-25 00:15:52,053 INFO Calculando t-SNE 3D para char_wb_2_4, min_df=2, seed=42


2026-10-25 00:16:16,982 INFO Calculando t-SNE 2D para char_wb_2_4, min_df=5, seed=42


...

2026-10-25 00:42:13,998 INFO Calculando t-SNE 3D para word_2_5, min_df=5, seed=42


2026-10-25 00:42:57,237 INFO Calculando t-SNE 2D para word_2_5, min_df=10, seed=42


2026-10-25 00:43:07,598 INFO Calculando t-SNE 3D para word_2_5, min_df=10, seed=42


2026-10-25 00:55:58,451 INFO Imagens t-SNE 2D e gr?ficos HTML 3D salvos: 3520


2026-10-25 00:55:58,481 INFO Relatorios e imagens t-SNE salvos em C:\Users\samuel.simao\Sources\Nofis-Classifier\reports\tfidf_dbscan\202501-202512_20261025T031512Z159795


## 7. Interpretacao??o

- `noise_fraction` alta significa que muitas descri??es n?o entraram em regi?es densas para os par?metros escolhidos.
- Poucos clusters ou um cluster dominante podem indicar que `eps` est? grande; quase tudo como ru?do pode indicar `eps` pequeno ou `min_samples` alto.
- Silhouette mais alta deve ser lida junto da fra??o de ru?do: uma pontua??o calculada em poucos itens pode ser enganosa.
- DBSCAN n?o atribui automaticamente novos produtos a clusters ap?s o ajuste e n?o define categorias sem revis?o humana.

- Os HTML em `tsne_3d_html/` cont?m uma proje??o t-SNE 3D que pode ser girada com o mouse e aberta sem depend?ncias externas.
